## Validation Datasets

The goal is to compare station data to ERA5 data and buoy data to GHRSST data to see how much error we are getting in a sample of regions in our models

From there, we can use that to figure out a reasonable "perturbation" for our model, changing 2m Temperature and SST accordingly to account for likely measurement error and use that to run ensemble models.

In [1]:
#import what you need
import xarray as xr #to read grib
import pandas as pd #for station data
import numpy as np #for subtracting arrays
from datetime import datetime, timedelta #to handle the dates
import statistics #to get mean and sd
from math import isnan #get rid of nas
from itertools import filterfalse #get rid of nas

In [12]:
## read in atmospheric data

#ERA5
era5_sfcMar = xr.load_dataset("../../01Data/SFC/ERA5_WRF_SFC_201703.grib", engine="cfgrib", decode_timedelta=True)
era5_sfcApr = xr.load_dataset("../../01Data/SFC/ERA5_WRF_SFC_201704.grib", engine="cfgrib", decode_timedelta=True)

#era5_plMar = xr.load_dataset("../../01Data/PLEV/ERA5_WRF_PLEV_201703.grib", engine="cfgrib")
#era5_plpr = xr.load_dataset("../../01Data/PLEV/ERA5_WRF_PLEV_201704.grib", engine="cfgrib")

#Station data
anglStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/AngletonLakeJackson/GHCNh_USW00012976_2017.psv", sep="|", low_memory=False)
beauStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Beaumont/GHCNh_USW00000313_2017.psv", sep="|", low_memory=False)
conrStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Conroe/GHCNh_USW00053902_2017.psv", sep="|", low_memory=False)
galvStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Galveston/GHCNh_USW00012923_2017.psv", sep="|", low_memory=False)
housStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Houston/GHCNh_USW00012975_2017.psv", sep="|", low_memory=False)

skipping variable: paramId==170 shortName='stl2'
Traceback (most recent call last):
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='depthBelowLandLayer' value=Variable(dimensions=(), data=0.0) new_value=Variable(dimensions=(), data=7.0)
skipping variable: paramId==183 shortName='stl3'
Traceback (most recent call last):
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 641, in dict_merge
    raise DatasetBuildE

In [11]:
## Get useful station information for later work

#Station locations
lats = [anglStation["LATITUDE"][0], beauStation["LATITUDE"][0], conrStation["LATITUDE"][0], 
        galvStation["LATITUDE"][0], housStation["LATITUDE"][0]]
lons = [anglStation["LONGITUDE"][0], beauStation["LONGITUDE"][0], conrStation["LONGITUDE"][0], 
        galvStation["LONGITUDE"][0], housStation["LONGITUDE"][0]]

## Prep some other variables for comparisons
FocalTimePts = [datetime.fromisoformat("2017-03-23T00:00:00"), datetime.fromisoformat("2017-04-01T00:00:00"), 
                datetime.fromisoformat("2017-04-03T00:00:00")]

#tolerance of difference in lat/lon from target to what we use for the mean. 
#We seem to have values every 0.25 units for each, so these are about half to find the closest
lonTol = 0.13 #unlikely to get an exact match
latTol = 0.13 #unlikely to get an exact match

TypeError: list indices must be integers or slices, not str

Start with the weather stations, going in alphabetical order

* Angleton Lake Jackson
* Beaumont
* Conroe
* Galveston
* Houston

In [7]:
##get avg temps
def getAvg(df, qc):

    avg_temps = []
    
    #loop through it
    for t in df.time.values:
    
        focalTime = pd.to_datetime(pd.Timestamp(t))
        # Subset data within the time tolerance
        subset = [
            entry['temperature'] for entry in anglStation
            if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == qc
        ]
    
        # Compute the average temperature if any entries matched
        if subset:
            avg = sum(subset) / len(subset)
        else:
            avg = float('nan')
            
        avg_temps.append(avg)
        
    return avg_temps

In [9]:
## Comparisons for Angleton Lake Jackson
# Find subset within the tolerance range
anglSubsetMar = era5_sfcMar.t2m.sel(
    latitude=slice(lats[0] + latTol, lats[0] - latTol),
    longitude=slice(lons[0] - lonTol, lons[0] + lonTol),
    time=slice(FocalTimePts[0], FocalTimePts[1])
)

anglSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[0] + latTol, lats[0] - latTol),
    longitude=slice(lons[0] - lonTol, lons[0] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

In [10]:
#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
anglStation = anglStation.to_dict('records')

mar_avg_temps = getAvg(anglSubsetMar, '5')
    
#get the difference between these time points and the closest station values
marDiffs = np.array([anglSubsetMar.mean(dim='latitude').values - 272.15]).flatten() - mar_avg_temps
angl_marDiffs = list(filterfalse(isnan, marDiffs))

# Store average temperatures - April
apr_avg_temps = []
mpr_avg_temps = getAvg(anglSubsetApr, '5')
    
#get the difference between these time points and the closest station values
aprDiffs = np.array([anglSubsetApr.mean(dim='latitude').values - 272.15]).flatten() - apr_avg_temps
angl_aprDiffs = list(filterfalse(isnan, aprDiffs))

AttributeError: 'list' object has no attribute 'to_dict'

In [6]:
#get some summary statistics
angl_avgDiff = statistics.mean([*angl_marDiffs, *angl_aprDiffs])
angl_medDiff = statistics.median([*angl_marDiffs, *angl_aprDiffs])

Beaumont

In [7]:
## Comparisons for Beaumont
# Find subset within the tolerance range
beauSubsetMar = era5_sfcMar.t2m.sel(
    latitude=slice(lats[1] + latTol, lats[1] - latTol),
    longitude=slice(lons[1] - lonTol, lons[1] + lonTol),
    time=slice(FocalTimePts[0], FocalTimePts[1])
)

beauSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[1] + latTol, lats[1] - latTol),
    longitude=slice(lons[1] - lonTol, lons[1] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=45)
#beauStation = beauStation.to_dict('records')

#loop through it
for t in beauSubsetMar.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in beauStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '1'
    ]

    #for debugging
    #print(f"Time: {focalTime}, Matched Temps: {subset}")

    
    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    mar_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
marDiffs = np.array([beauSubsetMar.mean(dim='latitude').values - 272.15]).flatten() - mar_avg_temps
beau_marDiffs = list(filterfalse(isnan, marDiffs))

# Store average temperatures - April
apr_avg_temps = []
tolerance= timedelta(minutes=30)

#loop through it
for t in beauSubsetApr.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in beauStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '1'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    apr_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
aprDiffs = np.array([beauSubsetApr.mean(dim='latitude').values - 272.15]).flatten() - apr_avg_temps
beau_aprDiffs = list(filterfalse(isnan, aprDiffs))

TypeError: string indices must be integers, not 'str'

In [ ]:
#get some summary statistics
beau_avgDiff = statistics.mean([*marDiffs.tolist(), *aprDiffs.tolist()])
beau_medDiff = statistics.median([*marDiffs.tolist(), *aprDiffs.tolist()])

Conroe

In [ ]:
## Comparisons for Conroe
# Find subset within the tolerance range
conrSubsetMar = era5_sfcMar.t2m.sel(
    latitude=slice(lats[2] + latTol, lats[2] - latTol),
    longitude=slice(lons[2] - lonTol, lons[2] + lonTol),
    time=slice(FocalTimePts[0], FocalTimePts[1])
)

conrSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[2] + latTol, lats[2] - latTol),
    longitude=slice(lons[2] - lonTol, lons[2] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
conrStation = conrStation.to_dict('records')

#loop through it
for t in conrSubsetMar.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in conrStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5.0'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    mar_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
marDiffs = np.array([conrSubsetMar.values - 272.15]).flatten() - mar_avg_temps
conr_marDiffs = list(filterfalse(isnan, marDiffs))

# Store average temperatures - April
apr_avg_temps = []
tolerance= timedelta(minutes=30)

#loop through it
for t in conrSubsetApr.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in conrStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5.0'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    apr_avg_temps.append(avg)
    

#get the difference between these time points and the closest station values
aprDiffs = np.array([conrSubsetApr.values - 272.15]).flatten() - apr_avg_temps
conr_aprDiffs = list(filterfalse(isnan, aprDiffs))

In [ ]:
#get some summary statistics
conr_avgDiff = statistics.mean([*conr_marDiffs, *conr_aprDiffs])
conr_medDiff = statistics.median([*conr_marDiffs, *conr_aprDiffs])

Galveston

In [ ]:
## Comparisons for Galveston
# Find subset within the tolerance range
galvSubsetMar = era5_sfcMar.t2m.sel(
    latitude=slice(lats[2] + latTol, lats[2] - latTol),
    longitude=slice(lons[2] - lonTol, lons[2] + lonTol),
    time=slice(FocalTimePts[0], FocalTimePts[1])
)

galvSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[2] + latTol, lats[2] - latTol),
    longitude=slice(lons[2] - lonTol, lons[2] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
galvStation = galvStation.to_dict('records')

#loop through it
for t in galvSubsetMar.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in galvStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5.0'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    mar_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
marDiffs = np.array([galvSubsetMar.mean(dim='latitude').values - 272.15]).flatten() - mar_avg_temps
galv_marDiffs = list(filterfalse(isnan, marDiffs))

# Store average temperatures - April
apr_avg_temps = []
tolerance= timedelta(minutes=30)

#loop through it
for t in galvSubsetApr.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in galvStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5.0'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    apr_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
aprDiffs = np.array([galvSubsetApr.mean(dim='latitude').values - 272.15]).flatten() - apr_avg_temps
galv_aprDiffs = list(filterfalse(isnan, aprDiffs))

In [ ]:
#get some summary statistics
galv_avgDiff = statistics.mean([*galv_marDiffs, *galv_aprDiffs])
galv_medDiff = statistics.median([*galv_marDiffs, *galv_aprDiffs])

Houston

In [ ]:
## Comparisons for Houston
# Find subset within the tolerance range
housSubsetMar = era5_sfcMar.t2m.sel(
    latitude=slice(lats[2] + latTol, lats[2] - latTol),
    longitude=slice(lons[2] - lonTol, lons[2] + lonTol),
    time=slice(FocalTimePts[0], FocalTimePts[1])
)

housSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[2] + latTol, lats[2] - latTol),
    longitude=slice(lons[2] - lonTol, lons[2] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
housStation = housStation.to_dict('records')

#loop through it
for t in housSubsetMar.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in housStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5.0'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    mar_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
marDiffs = np.array([housSubsetMar.mean(dim='latitude').values - 272.15]).flatten() - mar_avg_temps
hous_marDiffs = list(filterfalse(isnan, marDiffs))

# Store average temperatures - April
apr_avg_temps = []
tolerance= timedelta(minutes=30)

#loop through it
for t in housSubsetApr.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in housStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5.0'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    apr_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
aprDiffs = np.array([housSubsetApr.mean(dim='latitude').values - 272.15]).flatten() - apr_avg_temps
hous_aprDiffs = list(filterfalse(isnan, aprDiffs))

In [ ]:
#get some summary statistics
hous_avgDiff = statistics.mean([*hous_marDiffs, *hous_aprDiffs])
hous_medDiff = statistics.median([*hous_marDiffs, *hous_aprDiffs])

In [ ]:
means = [angl_avgDiff, beau_avgDiff, conr_avgDiff, galv_avgDiff, hous_avgDiff]
medians = [angl_medDiff, beau_medDiff, conr_medDiff, galv_medDiff, hous_medDiff]
OutputDF = pd.DataFrame(np.column_stack((means, medians)), columns=["mean", "median"], 
                        index=["Angleton", "Beaumont", "Conroe", "Galveston", "Houston"])
OutputDF.to_csv("../../03ProcessedData/ERA5Errors.csv")